# HeatShield AI — Colab Training

**กดรัน Cell เดียวแล้วรอ** — ได้ ZIP โมเดลใน Google Drive อัตโนมัติ

**ตั้งค่าก่อนรัน (ไม่บังคับ):**
- Colab Secrets → `CDSAPI_KEY`, `TMD_API_KEY` (ถ้าอยากใช้ ERA5/TMD)
- Runtime → Change runtime type → GPU (T4) เพื่อความเร็ว

**Output:** ZIP อยู่ที่ `MyDrive/heatshield/exports/HeatShield_artifacts_v*.zip`

In [ ]:
# ── CONFIG — แก้ตรงนี้ ──────────────────────────────────────────────────────
STAGE1_TRIALS = 60    # trials สำหรับ h=24 sweep
STAGE2_TRIALS = 100   # trials สำหรับ weak slots (ลดจาก 150 → 100, diminishing returns หลัง ~80)
STAGE3_TRIALS = 100   # trials สำหรับ h=6/12
RUN_STAGE3    = True  # False = ข้าม h=6/12
START_DATE    = "2021-01-01"
MODEL_VERSION = "v4"             # เปลี่ยนเป็น v5, v6 ... สำหรับ run ใหม่
BACKENDS      = "lgbm,catboost"  # "lgbm" เร็วกว่า; "lgbm,catboost" สำหรับ ensemble
FORCE_RETRAIN = True             # True = retrain เสมอ; False = ข้าม slot ที่มีแล้ว

In [ ]:
# ── SETUP: packages • Drive • repo ───────────────────────────────────────────
import os, sys, subprocess, hashlib, json, shutil, zipfile, time
import datetime
from collections import defaultdict
from pathlib import Path

# ── Colab secrets (optional — skip if not set) ────────────────────────────────
try:
    from google.colab import userdata as _ud
    for _k in ("CDSAPI_KEY", "TMD_API_KEY"):
        _v = _ud.get(_k)
        if _v:
            os.environ.setdefault(_k, _v)
except Exception:
    pass

# ── Google Drive mount ────────────────────────────────────────────────────────
try:
    from google.colab import drive as _drive
    _drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    pass  # running locally outside Colab

DRIVE_ROOT   = Path("/content/drive/MyDrive/heatshield")
VERSIONS_DIR = DRIVE_ROOT / "versions"
EXPORTS_DIR  = DRIVE_ROOT / "exports"
for _d in (DRIVE_ROOT, VERSIONS_DIR, EXPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ── Clone / pull repo ─────────────────────────────────────────────────────────
REPO = Path("/content/heatshield-backend")
_REPO_URL = "https://github.com/orbitorls/Heat-wave-backend.git"  # ← your fork
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", _REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO))

# ── Hash-based dep install (skips if requirements unchanged) ──────────────────
_REQ_FILE = REPO / "requirements.txt"
_SIG_FILE = DRIVE_ROOT / ".req_sig"
_req_hash = hashlib.md5(_REQ_FILE.read_bytes()).hexdigest()
if not _SIG_FILE.exists() or _SIG_FILE.read_text().strip() != _req_hash:
    print("Installing dependencies...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(_REQ_FILE)],
        check=True,
    )
    _SIG_FILE.write_text(_req_hash)
    print("Deps installed and cached.")
else:
    print("Deps up-to-date (cached).")

# ── Sanity: verify key GPU-fix is present ─────────────────────────────────────
from app.ml.forecast.backends.lgbm_backend import _sanitize_lgbm_params_for_device
print("_sanitize_lgbm_params_for_device ✓  (max_bin GPU-clamp active)")

In [ ]:
# ── DEVICE + HELPERS ──────────────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def _detect_device() -> str:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5,
        )
        if r.returncode == 0 and r.stdout.strip():
            return "gpu"
    except Exception:
        pass
    return "cpu"

DEVICE    = _detect_device()
N_WORKERS = 1   # GPU VRAM is shared; keep stations sequential
END_DATE  = datetime.date.today().isoformat()

print(f"Device : {DEVICE.upper()}")
print(f"Workers: {N_WORKERS}")
print(f"Window : {START_DATE} → {END_DATE}")

# ── Drive ↔ local model symlink ───────────────────────────────────────────────
DRIVE_MODELS = DRIVE_ROOT / "models"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
LOCAL_MODELS  = REPO / "app" / "models" / "forecast_v3"
LOCAL_MODELS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_MODELS.is_symlink():
    pass  # already linked
elif LOCAL_MODELS.exists():
    shutil.copytree(LOCAL_MODELS, DRIVE_MODELS, dirs_exist_ok=True)
    shutil.rmtree(LOCAL_MODELS)
    LOCAL_MODELS.symlink_to(DRIVE_MODELS)
else:
    LOCAL_MODELS.symlink_to(DRIVE_MODELS)

# ── Shared run state ──────────────────────────────────────────────────────────
RUNS_ROOT = DRIVE_ROOT / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR   = DRIVE_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
RUN_IDS:  list[str]  = []
_RID_SET: set[str]   = set()
TIMINGS:  list[dict] = []

# ── _train() helper ───────────────────────────────────────────────────────────
def _train(run_id: str, *, trials: int, horizons: str = "24",
           station: str = "all", model_version: str = MODEL_VERSION,
           force: bool = False) -> str:
    if run_id in _RID_SET:
        print(f"[skip] {run_id} already done this session")
        return run_id
    _RID_SET.add(run_id)
    (RUNS_ROOT / run_id).mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "scripts/train_forecast.py",
        "--horizons",      horizons,
        "--trials",        str(trials),
        "--backends",      BACKENDS,
        "--model-version", model_version,
        "--run-id",        run_id,
        "--device",        DEVICE,
    ]
    if station != "all":
        cmd += ["--station", station]
    if force:
        cmd.append("--force")
    print(f"\n>>> {' '.join(cmd)}")
    t0   = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    elapsed = time.time() - t0
    TIMINGS.append({"run_id": run_id, "seconds": round(elapsed, 1)})
    if run_id not in RUN_IDS:
        RUN_IDS.append(run_id)
    if proc.returncode != 0:
        print(f"[warn] {run_id} exited with code {proc.returncode}")
    return run_id

In [ ]:
# ── DATA: ingest NASA POWER observations ─────────────────────────────────────
print("\n" + "#"*60)
print("# DATA — ingest NASA POWER")
print("#"*60)

# Copy parquet store to /tmp to avoid Drive NFS latency during training
_drive_data = DRIVE_ROOT / "data"
_tmp_data   = Path("/tmp/heatshield_data")
if _drive_data.exists() and not _tmp_data.exists():
    print("Copying data to /tmp for faster I/O...")
    shutil.copytree(_drive_data, _tmp_data)
    _repo_data = REPO / "data"
    if _repo_data.is_symlink():
        _repo_data.unlink()
    elif _repo_data.exists():
        shutil.rmtree(_repo_data)
    _repo_data.symlink_to(_tmp_data)
    print("Data cached at /tmp.")

_r = subprocess.run(
    [sys.executable, "scripts/ingest_all.py",
     "--source", "nasa_power", "--start", START_DATE, "--end", END_DATE],
    capture_output=True, text=True,
)
if _r.returncode != 0:
    print("[warn] ingest_all.py exited", _r.returncode)
    print(_r.stderr[-2000:])
else:
    for _line in _r.stdout.splitlines():
        if any(kw in _line.lower() for kw in ("row", "station", "done", "skip", "warn")):
            print(_line)
    print("Ingest OK.")

In [ ]:
# ── STAGE 1: h=24 sweep (all stations) ───────────────────────────────────────
print("\n" + "#"*60)
print("# STAGE 1 — h=24 sweep (all stations)")
print("#"*60)
s1_id = _train("stage1", trials=STAGE1_TRIALS, horizons="24",
               model_version=MODEL_VERSION, force=FORCE_RETRAIN)

In [ ]:
# ── STAGE 2: refine weak slots ───────────────────────────────────────────────
print("\n" + "#"*60)
print("# STAGE 2 — refine weak slots")
print("#"*60)
lb1 = RUNS_ROOT / s1_id / "leaderboard.json"
if lb1.exists():
    weak_by_station: dict[str, set[int]] = defaultdict(set)
    for r in json.loads(lb1.read_text()):
        status = str(r.get("status", "")).lower()
        if status in {"not_ready", "candidate", "skipped_existing"} or r.get("skill_score") is None:
            sid, h = r.get("station"), r.get("horizon_h")
            if sid and h is not None:
                weak_by_station[str(sid)].add(int(h))
    n_weak = sum(len(v) for v in weak_by_station.values())
    print(f"Weak slots: {n_weak}")
    for sid in sorted(weak_by_station):
        hz = ",".join(str(h) for h in sorted(weak_by_station[sid]))
        _train(f"stage2_{sid}", trials=STAGE2_TRIALS, station=sid,
               horizons=hz, force=True, model_version=MODEL_VERSION)
else:
    print("[warn] No leaderboard from Stage 1 — skipping Stage 2")

In [ ]:
# ── STAGE 3: h=6/12 for ready stations (optional) ───────────────────────────
if RUN_STAGE3:
    print("\n" + "#"*60)
    print("# STAGE 3 — h=6/12 for ready stations")
    print("#"*60)
    all_rows: list[dict] = []
    for rid in RUN_IDS:
        lb = RUNS_ROOT / rid / "leaderboard.json"
        if lb.exists():
            all_rows.extend(json.loads(lb.read_text()))
    latest: dict = {}
    for r in all_rows:
        latest[(r.get("station"), r.get("horizon_h"))] = r
    ready = sorted({
        sid for (sid, h), r in latest.items()
        if h == 24 and str(r.get("status", "")).lower() in {"ready", "skipped_existing"}
    })
    print(f"Ready stations for h=6/12: {ready}")
    for sid in ready:
        _train(f"stage3_{sid}", trials=STAGE3_TRIALS, station=sid,
               horizons="6,12", model_version=MODEL_VERSION)
else:
    print("RUN_STAGE3=False — skipping h=6/12")

In [ ]:
# ── EXPORT: snapshot + ZIP ───────────────────────────────────────────────────
print("\n" + "#"*60)
print("# EXPORT — building ZIP snapshot")
print("#"*60)

def _next_ver(base: Path) -> int:
    vs = [int(d.name[1:]) for d in base.iterdir()
          if d.is_dir() and d.name.startswith("v") and d.name[1:].isdigit()]
    return (max(vs) + 1) if vs else 1

slot_meta: dict = {}
warnings:  list[str] = []
for rid in RUN_IDS:
    lb = RUNS_ROOT / rid / "leaderboard.json"
    if not lb.exists():
        warnings.append(f"missing_lb:{rid}")
        continue
    for r in json.loads(lb.read_text()):
        sid, h = r.get("station"), r.get("horizon_h")
        if sid in (None, "all") or h is None:
            continue
        slot_meta[(str(sid), int(h))] = {
            "station": str(sid), "horizon_h": int(h),
            "backend": r.get("backend"), "status": r.get("status"),
            "source_run_id": rid,
        }

snap_ver = f"v{_next_ver(VERSIONS_DIR)}"
snap_dir = VERSIONS_DIR / snap_ver
snap_dir.mkdir(parents=True, exist_ok=False)

for (sid, h) in sorted(slot_meta):
    src = LOCAL_MODELS / sid / f"h{h}"
    dst = snap_dir / sid / f"h{h}"
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        warnings.append(f"missing:{sid}:h{h}")

cm = LOCAL_MODELS / "choice_matrix.json"
if cm.exists():
    shutil.copy2(cm, snap_dir / "choice_matrix.json")

# Collect CatBoost bundle metadata
catboost_slots: list[dict] = []
for (sid, h) in sorted(slot_meta):
    cb_bundle = LOCAL_MODELS / sid / f"h{h}" / "catboost" / "bundle.json"
    if cb_bundle.exists():
        try:
            cb_meta = json.loads(cb_bundle.read_text())
            catboost_slots.append({
                "station": sid, "horizon_h": h,
                "backend": cb_meta.get("backend_name", "catboost_quantile"),
                "mae":     cb_meta.get("metrics", {}).get("mae"),
            })
        except Exception:
            pass

manifest = {
    "created_at_utc":   datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "snapshot_version": snap_ver,
    "model_version":    MODEL_VERSION,
    "backends":         BACKENDS,
    "run_ids":          RUN_IDS,
    "slots":            [slot_meta[k] for k in sorted(slot_meta)],
    "catboost_slots":   catboost_slots,
    "warnings":         warnings,
    "timings":          TIMINGS,
}
(snap_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

stamp    = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_name = f"HeatShield_artifacts_{snap_ver}_{stamp}.zip"
zip_path = EXPORTS_DIR / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fp in sorted(snap_dir.rglob("*")):
        if fp.is_file():
            zf.write(fp, (Path("models") / snap_ver / fp.relative_to(snap_dir)).as_posix())
    for rid in RUN_IDS:
        run_dir = RUNS_ROOT / rid
        if run_dir.exists():
            for fp in sorted(run_dir.rglob("*")):
                if fp.is_file():
                    zf.write(fp, (Path("eval") / rid / fp.relative_to(run_dir)).as_posix())
    zf.writestr("manifest.json", json.dumps(manifest, indent=2, ensure_ascii=False))

mb        = zip_path.stat().st_size / 1024 / 1024
total_min = sum(t["seconds"] for t in TIMINGS) / 60

print(f"""
{'='*60}
DONE
  Snapshot   : {snap_ver}
  Model Ver  : {MODEL_VERSION}
  Backends   : {BACKENDS}
  LGBM slots : {len(slot_meta)}
  CB slots   : {len(catboost_slots)}
  ZIP        : {zip_path}
  Size       : {mb:.1f} MB
  Total time : {total_min:.0f} min
{'='*60}

ดาวน์โหลด ZIP จาก Google Drive:
  MyDrive/heatshield/exports/{zip_name}
""")

if warnings:
    print("Warnings:")
    for w in warnings:
        print(" -", w)

## วิธีใช้โมเดล (Windows)

```powershell
# 1. Download ZIP จาก MyDrive/heatshield/exports/

# 2. Extract + promote
$ZIP = Get-ChildItem .\HeatShield_artifacts_v*.zip | Sort LastWriteTime -Desc | Select -First 1
Expand-Archive $ZIP.FullName -DestinationPath .\artifact_unpack -Force
$VER = Get-ChildItem ".\artifact_unpack\models" -Directory | Sort Name -Desc | Select -First 1
Remove-Item app\models\forecast_v3 -Recurse -Force -ErrorAction SilentlyContinue
New-Item -ItemType Directory -Force -Path app\models\forecast_v3 | Out-Null
Copy-Item "$($VER.FullName)\*" "app\models\forecast_v3\" -Recurse -Force

# 3. Verify + evaluate
python -c "from app.ml.registry import load_latest_v3; print(load_latest_v3('BKK_01', 24).backend_name)"
python scripts/evaluate_model.py
```